# Chapter 4: Inspect ordinary search and its continuation

Chapter 4 compares search methods under a common budget. This notebook reads the three ordinary arms of the fresh run (twinning initialization, random initialization and equal-budget continuation) and reports each run's evaluated history and primary selection. The optimizer searches within the admitted representation. Compare ordinary initializations and the equal-additional-budget continuation before examining structural change.

**Evidence.** Run 20260923T220909Z repeated the regional study end to end on two machines. `spark-fe12` ran numerical discovery with three new optimizer seeds (74101 to 74103), one live structural proposal and a fresh 200,000-observation confirmation sample. `spark-ef84` ran 70 live answer tests and nine report drafts with `claude-sonnet-5`. The population and frozen logistic predictor are those of the chapter companions; the seeds, proposal and confirmation sample are new, so these numbers should not be combined with the historical run's.

## Setup and source checks

The setup cell locates the data root, the directory whose relative layout the evidence records use. It uses `MODEL_VALIDATION_ROOT` when that variable is set. Otherwise it searches for a repository checkout containing the current directory and then for the copy installed with the `aiv` package. `R` is this run's folder under the data root, and `read` loads a JSON file from it.

`reference.json` was exported from the run's regional results and records their SHA-256 digest. The final assertion recomputes that digest, so every value used below comes from the unchanged results of this run. This notebook reads recorded outputs; rerunning it makes no language-model call.

In [1]:
from pathlib import Path
import json, hashlib, os, platform
import numpy as np
import pandas as pd

# Locate the data root: the directory whose relative layout the evidence records use.
MARKER = "book/notebooks/regional_case/companion.py"

def find_data_root():
    configured = os.environ.get("MODEL_VALIDATION_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / MARKER).is_file():
            raise FileNotFoundError(f"MODEL_VALIDATION_ROOT must contain {MARKER}")
        return root, "MODEL_VALIDATION_ROOT"
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / MARKER).is_file():
            return path, "repository checkout"
    try:
        from aiv.companions import locate
    except ImportError:
        raise FileNotFoundError(
            "Companion data not found. Install the aiv package, open the notebook "
            "inside the repository checkout, or set MODEL_VALIDATION_ROOT.") from None
    return locate()

ROOT, DATA_SOURCE = find_data_root()
LIVE = ROOT / "book/experiments/regional_live"
R = ROOT / "book/experiments/regional_live/runs/20260923T220909Z"
print("Python:", platform.python_version())
print("Data root found from:", DATA_SOURCE)
print("Run:", R.relative_to(ROOT))
read=lambda p: json.loads((R/p).read_text())
results=read('regional/results.json')
reference=read('reference.json')
seeds=reference['seeds']
assert hashlib.sha256((R/'regional/results.json').read_bytes()).hexdigest()==reference['source_sha256']


Python: 3.12.3
Data root found from: repository checkout
Run: book/experiments/regional_live/runs/20260923T220909Z


## Ordinary search and continuation

For each seed and arm the cell loads the search history, counts its candidate records and reports the discovery objectives `[contrast, share]` of its primary selection. Continuation inherits the twinning run's 400 records and adds 400 of its own, so its history has 800 records for the same additional budget.

In [2]:
rows=[]
for seed in seeds:
 for arm in ['twinning','random','continuation']:
  run=read(f'regional/{arm}_{seed}.json')
  rows.append(dict(seed=seed,arm=arm,scored=len(run['records']),objectives=run['primary']['objectives']))
display(pd.DataFrame(rows))

,seed,arm,scored,objectives
0,74101,twinning,400,"[0.06686019793661731, 0.103525]"
1,74101,random,400,"[0.06686019793661731, 0.103525]"
2,74101,continuation,800,"[0.06686019793661731, 0.103525]"
3,74102,twinning,400,"[0.06484871872744419, 0.102525]"
4,74102,random,400,"[0.06484871872744419, 0.102525]"
5,74102,continuation,800,"[0.06484871872744419, 0.102525]"
6,74103,twinning,400,"[0.062222290413305445, 0.111725]"
7,74103,random,400,"[0.06151516900522714, 0.116525]"
8,74103,continuation,800,"[0.07233707886575506, 0.1034]"


**Read the output.** At seeds 74101 and 74102 all three arms selected the same discovery region: contrast 0.066860 at share 0.103525, and contrast 0.064849 at share 0.102525. The additional continuation budget found no eligible candidate with a larger contrast at those seeds. At seed 74103 continuation selected a narrower region with a larger discovery contrast, 0.072337 at share 0.1034, than twinning (0.062222) or random initialization (0.061515). Discovery contrasts were maximized on the sample that guided search, so these values rank candidates within a run; the confirmation values in the first notebook assess them.